In [5]:
# ===================================================
# VALIDACIÓN DE HORIZONTE DE FORECAST (1h vs 3h)
# ===================================================

import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ===================================================
# 1. CARGA DE DATOS
# ===================================================
file_path = "../data/raw/rappi_delivery_case_data.xlsx"

# leer hojas del Excel
df = pd.read_excel(file_path, sheet_name="RAW_DATA")

df["datetime"] = (
    pd.to_datetime(df["DATE"]) +
    pd.to_timedelta(df["HOUR"], unit="h")
)

# ===================================================
# 2. FUNCIÓN DE EVALUACIÓN
# ===================================================
def walk_forward_mae(series, train_frac=0.75, n_eval=50):

    n = len(series)
    train_end = int(n * train_frac)

    errors_1h = []
    errors_3h = []

    for i in range(train_end, min(n - 3, train_end + n_eval)):

        try:
            model = ExponentialSmoothing(
                series.iloc[:i],
                trend='add',
                seasonal='add',
                seasonal_periods=24
            ).fit(
                smoothing_level=0.3,
                smoothing_trend=0.1,
                smoothing_seasonal=0.1,
                optimized=False
            )

            forecast = model.forecast(3)

            errors_1h.append(abs(forecast.iloc[0] - series.iloc[i]))
            errors_3h.append(abs(forecast.iloc[2] - series.iloc[i + 2]))

        except:
            continue

    return np.mean(errors_1h), np.mean(errors_3h)

# ===================================================
# 3. EVALUACIÓN POR ZONA
# ===================================================
results = []

for zone in df["ZONE"].unique():

    series = (
        df[df["ZONE"] == zone]
        .groupby("datetime")["ORDERS"]
        .sum()
        .asfreq("h", fill_value=0)
    )

    mae_1h, mae_3h = walk_forward_mae(series)

    results.append({
        "ZONE": zone,
        "MAE_1h": mae_1h,
        "MAE_3h": mae_3h
    })

# ===================================================
# 4. RESULTADOS
# ===================================================
mae_df = pd.DataFrame(results)

mae_df["degradacion_pct"] = (
    (mae_df["MAE_3h"] - mae_df["MAE_1h"]) / mae_df["MAE_1h"] * 100
).round(1)

mae_df = mae_df.sort_values("degradacion_pct", ascending=False)

# ===================================================
# 5. OUTPUT
# ===================================================
print(mae_df.to_string(index=False))

print("\nResumen:")
print(f"MAE 1h promedio: {mae_df.MAE_1h.mean():.2f}")
print(f"MAE 3h promedio: {mae_df.MAE_3h.mean():.2f}")
print(f"Degradación promedio: {mae_df.degradacion_pct.mean():.1f}%")

               ZONE   MAE_1h   MAE_3h  degradacion_pct
        San Nicolás 0.936548 1.155352             23.4
      Mitras Centro 1.112688 1.325863             19.2
     Santa Catarina 0.693108 0.808103             16.6
      Independencia 0.859919 0.989553             15.1
              La Fe 0.931508 1.045029             12.2
     Apodaca Centro 0.903687 1.012390             12.0
             Centro 1.942861 2.173892             11.9
           Santiago 0.543848 0.606252             11.5
MTY_Apodaca_Huinalá 0.479130 0.529825             10.6
      MTY_Guadalupe 0.987753 1.085896              9.9
 Carretera Nacional 0.692429 0.756962              9.3
          San Pedro 1.131060 1.231735              8.9
   Cumbres Poniente 0.952894 1.023436              7.4
           Escobedo 0.843447 0.861658              2.2

Resumen:
MAE 1h promedio: 0.93
MAE 3h promedio: 1.04
Degradación promedio: 12.2%


In [4]:
# ===================================================
# JUSTIFICACIÓN DE UMBRALES DE LLUVIA POR ZONA
# ===================================================
# Objetivo:
# Determinar a partir de datos históricos:
# - Qué nivel de precipitación dispara riesgo operacional
# - Si el umbral es homogéneo o específico por zona
#
# Metodología:
# 1. Construir ratio operacional
# 2. Definir evento de saturación (ratio > 1.8)
# 3. Discretizar lluvia en bins (cuantiles)
# 4. Estimar P(saturación | lluvia)
# 5. Detectar threshold donde el riesgo supera un nivel crítico
# ===================================================

import pandas as pd
import numpy as np

# ===================================================
# 1. CARGA DE DATOS
# ===================================================
file_path = "../data/raw/rappi_delivery_case_data.xlsx"

# leer hojas del Excel
df = pd.read_excel(file_path, sheet_name="RAW_DATA")

df["datetime"] = (
    pd.to_datetime(df["DATE"]) +
    pd.to_timedelta(df["HOUR"], unit="h")
)

df = df.sort_values(["ZONE", "datetime"])

# ===================================================
# 2. FEATURE ENGINEERING
# ===================================================

# evitar división por cero
df["CONNECTED_RT"] = df["CONNECTED_RT"].replace(0, np.nan)

# ratio operacional
df["RATIO"] = df["ORDERS"] / df["CONNECTED_RT"]
df["RATIO"] = df["RATIO"].fillna(0)

# evento crítico
df["IS_SATURATED"] = (df["RATIO"] > 1.8).astype(int)

# ===================================================
# 3. BINNING DE LLUVIA
# ===================================================
df["rain_bin"] = "no_rain"

mask = df["PRECIPITATION_MM"] > 0

df.loc[mask, "rain_bin"] = pd.qcut(
    df.loc[mask, "PRECIPITATION_MM"],
    q=4,
    duplicates="drop"
).astype(str)

# ===================================================
# 4. CURVA LLUVIA → RIESGO
# ===================================================
rain_curve = df.groupby(["ZONE", "rain_bin"]).agg(
    prob_saturation=("IS_SATURATED", "mean"),
    count=("IS_SATURATED", "count"),
    avg_rain=("PRECIPITATION_MM", "mean")
).reset_index()

# eliminar bins con poco soporte
rain_curve = rain_curve[rain_curve["count"] >= 5]

# ordenar bins correctamente
def get_bin_start(x):
    if x == "no_rain":
        return 0
    try:
        return float(x.split(",")[0].replace("(", "").replace("[", ""))
    except:
        return 0

rain_curve["order"] = rain_curve["rain_bin"].apply(get_bin_start)
rain_curve = rain_curve.sort_values(["ZONE", "order"])

# ===================================================
# 5. DETECCIÓN DE UMBRAL POR ZONA
# ===================================================
THRESHOLD_PROB = 0.20  # riesgo mínimo aceptable

thresholds = []

for zone in rain_curve["ZONE"].unique():

    temp = rain_curve[rain_curve["ZONE"] == zone]

    # buscar primer punto donde probabilidad supera el threshold
    temp_thr = temp[temp["prob_saturation"] >= THRESHOLD_PROB]

    if len(temp_thr) > 0:
        first = temp_thr.iloc[0]

        thresholds.append({
            "ZONE": zone,
            "threshold_mm": round(first["order"], 2),
            "prob_saturation": round(first["prob_saturation"], 2),
            "avg_rain_bin": round(first["avg_rain"], 2)
        })

    else:
        thresholds.append({
            "ZONE": zone,
            "threshold_mm": 0.7,  # fallback conservador
            "prob_saturation": 0.3,
            "avg_rain_bin": 0.7
        })

thresholds_df = pd.DataFrame(thresholds)

# ===================================================
# 6. VALIDACIÓN: ¿UMBRAL GLOBAL O POR ZONA?
# ===================================================
print("\n===== UMBRALES POR ZONA =====\n")
print(thresholds_df.sort_values("threshold_mm").to_string(index=False))

print("\n===== DISPERSIÓN DE UMBRALES =====")
print(f"Min threshold: {thresholds_df['threshold_mm'].min():.2f}")
print(f"Max threshold: {thresholds_df['threshold_mm'].max():.2f}")
print(f"Std dev: {thresholds_df['threshold_mm'].std():.2f}")

# ===================================================
# 7. CONCLUSIÓN AUTOMÁTICA
# ===================================================
if thresholds_df["threshold_mm"].std() > 0.2:
    print("\nConclusión: Los umbrales NO son homogéneos → deben ser específicos por zona.")
else:
    print("\nConclusión: Los umbrales son relativamente homogéneos → se podría usar un threshold global.")


===== UMBRALES POR ZONA =====

               ZONE  threshold_mm  prob_saturation  avg_rain_bin
     Apodaca Centro           0.7              0.4          2.14
 Carretera Nacional           0.7              0.5          2.14
             Centro           0.7              0.4          2.14
   Cumbres Poniente           0.7              0.3          2.14
           Escobedo           0.7              0.4          2.14
      Independencia           0.7              0.4          2.14
              La Fe           0.7              0.3          2.14
MTY_Apodaca_Huinalá           0.7              0.3          2.14
      MTY_Guadalupe           0.7              0.4          2.14
      Mitras Centro           0.7              0.4          2.14
        San Nicolás           0.7              0.3          2.14
          San Pedro           0.7              0.4          2.14
     Santa Catarina           0.7              0.4          2.14
           Santiago           0.7              0.6        